# 08 - Explainability (SHAP + LIME)

* Global feature importance via SHAP TreeExplainer
* Per-prediction local explanations via LIME
* Plain-language security tips per feature


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.utils import get_config, setup_logging, load_joblib
from src.features import FeatureEngineer, FeatureSelector
from src.explainability import GlobalExplainer
setup_logging()
cfg = get_config()

X_tr = pd.read_csv(cfg.processed_data_dir / "X_train.csv")
y_tr = pd.read_csv(cfg.processed_data_dir / "y_train.csv").iloc[:,0]
eng = FeatureEngineer.load(cfg.preprocessing_dir / "feature_engineer.joblib")
sel = load_joblib(cfg.features_dir / "feature_selector.joblib")
X_tr_e = sel.transform(eng.transform(X_tr))
model = load_joblib(cfg.trained_models_dir / "best_model.joblib")
print(f"Best model: {model.__class__.__name__}")


## 1. SHAP global importance

In [ ]:
expl = GlobalExplainer(model=model, background=X_tr_e,
                       feature_names=list(X_tr_e.columns))
importance = expl.feature_importance(sample_size=200)
importance.head(20).plot(kind='barh', figsize=(8,7), color='#00E5A8')
plt.gca().invert_yaxis()
plt.title('SHAP global feature importance (mean |SHAP|)')
plt.tight_layout(); plt.show()


## 2. SHAP summary plot

In [ ]:
try:
    import shap
    shap_values, X_bg = expl.compute_shap_values(sample_size=200)
    shap.summary_plot(shap_values, X_bg, feature_names=list(X_tr_e.columns),
                      show=True)
except Exception as e:
    print(f"SHAP summary plot failed: {e}")


## 3. Local LIME explanation for a single record

In [ ]:
from src.explainability import LocalExplainer, build_security_tip
record = {f: int(X_tr.iloc[0][f]) for f in cfg.features}
print("Sample record (first 5 features):", dict(list(record.items())[:5]))

le = LocalExplainer(model, X_tr_e, list(X_tr_e.columns),
                    is_xgb_like=False)
label, weights = le.explain_instance(pd.Series(record), num_features=10)
print(f"Predicted: {label}")
for feat, w in weights:
    print(f"  {feat}: {w:+.4f}")


## 4. Plain-language security tips

In [ ]:
tips_cfg = cfg.prediction_schema.feature_security_tips
for fname in list(cfg.features)[:5]:
    val = record[fname]
    tip = build_security_tip(fname, val, tips_cfg)
    print(f"  {fname}={val} -> {tip}")


## Summary

SHAP global importance typically surfaces `SSLfinal_State`,
`URL_of_Anchor`, and `web_traffic` as the top three drivers - matching
the published UCI literature. LIME provides per-prediction explanations,
and the `feature_security_tips` map in `prediction_schema.yaml`
translates each feature's value into a plain-language security tip the
end-user can act on.
